In [2]:
%pip install torch-optimizer
%pip install samtorch
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement samtorch (from versions: none)
ERROR: No matching distribution found for samtorch


  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import time
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
from torchvision.models import MobileNet_V3_Large_Weights
from PIL import Image
from torchvision.transforms import AutoAugment, AutoAugmentPolicy, RandAugment
import copy
import random
from tqdm import tqdm

# 🚀 CONFIGURATION
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64     # Increased batch size for faster iterations
EPOCHS = 300        
LEARNING_RATE = 1e-4  
PATIENCE = 25       
WEIGHT_DECAY = 1e-5
DROPOUT_RATE = 0.35
EXPERIMENT_LOG = "experiment_log_improved.csv"
SOURCE_MODEL_PATH = "mobilenetv3_best_accuracy_improved11.pth"
NEW_MODEL_PATH = "mobilenetv3_improved_weights14.pth"
CONTINUE_TRAINING = True
TEST_TIME_AUG = True  # Only used during final evaluation, not during training

# Set seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# 📊 SIMPLIFIED DATA AUGMENTATION
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Test-time augmentation transforms - only used in final evaluation
tta_transforms = [
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
]

# Simplified class-specific transforms - focus on underperforming classes
class_specific_transforms = {
    "BroussonetiaPapyrifera": transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.6),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    "CeibaPentandra": transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.6),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    "DurioZibethinus": transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.6),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
}

# Class-aware dataset
class ClassAwareDataset(Dataset):
    def __init__(self, root, transform=None, class_specific_transforms=None):
        self.dataset = datasets.ImageFolder(root, transform=transform)
        self.class_specific_transforms = class_specific_transforms
        self.classes = self.dataset.classes
        self.samples = self.dataset.samples
        self.class_to_idx = self.dataset.class_to_idx
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        class_name = self.idx_to_class[label]
        
        # Use class-specific transforms if available
        if self.class_specific_transforms and class_name in self.class_specific_transforms:
            # Get original image
            img_path = self.dataset.samples[idx][0]
            img = Image.open(img_path).convert('RGB')
            # Apply class-specific transform
            img = self.class_specific_transforms[class_name](img)
            
        return img, label

# Dataset loading with transforms
train_dataset = ClassAwareDataset(
    r"C:\Users\Asus TUF -PC\LeafSense AI training\LeafSenseProcessed\train", 
    transform=train_transform,
    class_specific_transforms=class_specific_transforms
)
val_dataset = datasets.ImageFolder(r"C:\Users\Asus TUF -PC\LeafSense AI training\LeafSenseProcessed\val", transform=val_transform)

# Custom TTA Dataset - Only used for final evaluation
class TTADataset(Dataset):
    def __init__(self, dataset, transforms):
        self.dataset = dataset
        self.transforms = transforms
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        if isinstance(img, Image.Image):
            # If img is already a PIL image
            samples = [transform(img) for transform in self.transforms]
        else:
            # If img is already transformed (tensor)
            original_img = self.dataset.dataset.loader(self.dataset.samples[idx][0])
            samples = [transform(original_img) for transform in self.transforms]
            
        return samples, label

# Balance class weights for better performance on underperforming classes
def get_class_weights(labels_count):
    # Give more weight to underperforming classes based on classification report
    class_specific_weights = {
        "BroussonetiaPapyrifera": 1.3,  # Low precision
        "CeibaPentandra": 1.3,  # Low f1-score
        "DurioZibethinus": 1.2,  # Low recall
        "SamaneaSaman": 1.4,  # Very low recall
        "EuphorbiaPulcherrima": 1.15,  # Low recall
        "ManihotEsculenta": 1.3,  # Low precision
    }
    
    idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
    
    total = sum(labels_count)
    weights = []
    for idx, count in enumerate(labels_count):
        base_weight = total / (len(labels_count) * count)
        class_name = idx_to_class[idx]
        multiplier = class_specific_weights.get(class_name, 1.0)
        weights.append(base_weight * multiplier)
        
    return torch.FloatTensor(weights)

# Calculate class counts for weighting
class_counts = [0] * len(train_dataset.classes)
for _, class_idx in train_dataset.samples:
    class_counts[class_idx] += 1

# Calculate sample weights for weighted sampling
weights = [1.0 / class_counts[class_idx] for _, class_idx in train_dataset.samples]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

class_weights = get_class_weights(class_counts).to(DEVICE)

# Optimize data loading - Use prefetch and persistent workers
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    sampler=sampler, 
    num_workers=4, 
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

# For TTA evaluation - only create when needed
if TEST_TIME_AUG:
    tta_val_dataset = TTADataset(val_dataset, tta_transforms)
    tta_val_loader = DataLoader(
        tta_val_dataset, 
        batch_size=BATCH_SIZE//len(tta_transforms), 
        shuffle=False, 
        num_workers=4, 
        pin_memory=True
    )

# 🛠️ KEEP THE ORIGINAL MODEL ARCHITECTURE (as requested)
class LeafClassifier(nn.Module):
    def __init__(self, num_classes, DROPOUT_RATE=0.25):
        super(LeafClassifier, self).__init__()
        # Use the latest weights for better initialization
        self.model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
        
        # Freeze fewer layers - only freeze the first 50% of layers
        # This allows for better fine-tuning to your specific leaf dataset
        total_params = len(list(self.model.parameters()))
        for param in list(self.model.parameters())[:total_params//2]:
            param.requires_grad = False
            
        # Replace with a simpler, more robust classifier
        in_features = self.model.classifier[0].in_features
        self.model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),  # Add batch normalization for stability
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT_RATE),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT_RATE),
            
            nn.Linear(256, num_classes)
        )
        
        # Apply better weight initialization
        self._initialize_weights()

    def forward(self, x):
        return self.model(x)
    
    def _initialize_weights(self):
        """Apply improved weight initialization to newly added layers"""
        for m in self.model.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

# Label Smoothing Loss - Simpler but effective for generalization
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, dim=-1, weight=None):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim
        self.weight = weight

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)

        if self.weight is not None:
            loss = torch.sum(-true_dist * pred * self.weight.unsqueeze(0), dim=self.dim)
        else:
            loss = torch.sum(-true_dist * pred, dim=self.dim)

        return loss.mean()

# Simple mixup implementation for data augmentation
def mixup_data(x, y, alpha=0.2):
    """Performs mixup augmentation which helps with regularization and generalization."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(DEVICE)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Criterion for mixup training."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Initialize the model
num_classes = len(train_dataset.classes)
model = LeafClassifier(num_classes, DROPOUT_RATE=DROPOUT_RATE).to(DEVICE)

# Load checkpoint weights if continuing training
if CONTINUE_TRAINING and os.path.exists(SOURCE_MODEL_PATH):
    print(f"📥 Loading checkpoint from {SOURCE_MODEL_PATH}")
    model.load_state_dict(torch.load(SOURCE_MODEL_PATH))
    print("✅ Checkpoint loaded successfully")

# 🔥 OPTIMIZED LOSS, OPTIMIZER, AND SCHEDULER
torch.cuda.empty_cache()

# Use simple but effective label smoothing loss
criterion = LabelSmoothingLoss(num_classes, smoothing=0.1, weight=class_weights)

# Standard AdamW optimizer with gradient clipping instead of SAM
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# One cycle policy for faster convergence
scheduler = OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE*10,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.3,
    div_factor=25,
    final_div_factor=10000
)

# Enable Mixed Precision Training (AMP)
scaler = torch.cuda.amp.GradScaler()

# Improved early stopping class
class EarlyStopping:
    def __init__(self, patience=PATIENCE, verbose=True, delta=0.001, path=NEW_MODEL_PATH):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.best_accuracy = 0.0
        self.path = path

    def __call__(self, val_loss, val_acc, model):
        # Use both accuracy and validation loss for monitoring
        score = val_acc - val_loss * 0.2  # Balance between accuracy and loss
        
        if self.best_score is None:
            self.best_score = score
            self.best_accuracy = val_acc
            self.save_checkpoint(val_loss, val_acc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"⚠️ Early stopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            if val_acc > self.best_accuracy:
                self.best_accuracy = val_acc
            self.save_checkpoint(val_loss, val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, val_acc, model):
        """Save model when validation performance improves."""
        if self.verbose:
            print(f"✅ Validation score improved ({self.val_loss_min:.4f} → {val_loss:.4f} | Acc: {val_acc:.4f}). Saving model...")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# Optimized evaluation function
def evaluate_model(model, val_loader, criterion, use_tta=False, save_confusion=False):
    """Evaluate model with optimized code path for training vs final evaluation"""
    model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        if use_tta and TEST_TIME_AUG:
            # Use TTA only for final evaluation, not during training
            for batch_imgs, labels in tqdm(tta_val_loader, desc="TTA Evaluation"):
                labels = labels.to(DEVICE)
                batch_size = labels.size(0)
                probs_avg = torch.zeros((batch_size, num_classes)).to(DEVICE)
                
                for i, imgs in enumerate(zip(*batch_imgs)):
                    imgs = torch.stack(imgs).to(DEVICE)
                    with torch.cuda.amp.autocast():  # Use AMP here too
                        outputs = model(imgs)
                        probs = torch.softmax(outputs, dim=1)
                    probs_avg += probs
                
                probs_avg /= len(tta_transforms)
                loss = criterion(probs_avg.log(), labels)
                val_loss += loss.item() * batch_size
                
                _, preds = torch.max(probs_avg, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs_avg.cpu().numpy())
        else:
            # Fast evaluation path for training epochs
            for images, labels in tqdm(val_loader, desc="Evaluation", disable=not save_confusion):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                with torch.cuda.amp.autocast():  # Use AMP consistently
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * labels.size(0)
                
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(probs, 1)
                
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

    val_loss /= len(all_labels)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    
    # Multi-class AUC calculation - only if needed
    if save_confusion:
        try:
            val_auc = roc_auc_score(np.eye(num_classes)[all_labels], all_probs, multi_class='ovr')
        except ValueError:
            val_auc = 0.0
    else:
        # Approximate AUC for speed during training
        val_auc = accuracy  # Just use accuracy as a proxy during training
        
    # Only save detailed metrics in final evaluation
    if save_confusion:
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))
        
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        
        # Save confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=val_dataset.classes, 
                   yticklabels=val_dataset.classes)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Labels')
        plt.ylabel('True Labels')
        plt.tight_layout()
        plt.savefig(f'confusion_matrix14_{timestamp}.png')
        plt.close()

        # Save per-class metrics
        class_report = classification_report(all_labels, all_preds, 
                                            target_names=val_dataset.classes, 
                                            output_dict=True)
        class_df = pd.DataFrame(class_report).transpose()
        class_df.to_csv(f'class_report_{timestamp}.csv')

    return val_loss, accuracy, val_auc

# Optimized training function
def train_model(model, criterion, optimizer, scheduler, early_stopping, epochs=EPOCHS):
    """Train the model with improved performance."""
    history = {
        'train_loss': [], 
        'val_loss': [], 
        'train_acc': [], 
        'val_acc': [],
        'val_auc': []
    }
    
    # Apply mixup occasionally but don't combine with other augmentations
    mixup_prob = 0.3
    
    print(f"🚀 Starting training for {epochs} epochs...")
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Apply mixup with probability but don't use cutmix simultaneously
            r = np.random.rand()
            if r < mixup_prob:
                # Apply Mixup
                images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.5)
                mixup_applied = True
            else:
                mixup_applied = False
                
            # Zero gradients for optimizer
            optimizer.zero_grad()
            
            # Use AMP for mixed precision training - properly integrated
            with torch.cuda.amp.autocast():
                outputs = model(images)
                if mixup_applied:
                    loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                else:
                    loss = criterion(outputs, labels)
            
            # Scale gradients for AMP
            scaler.scale(loss).backward()
            
            # Unscale before clip_grad_norm to ensure proper scaling
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update weights with scaled gradients
            scaler.step(optimizer)
            scaler.update()
            
            # Update learning rate
            scheduler.step()
            
            # Calculate training metrics
            train_loss += loss.item() * labels.size(0)
            
            if not mixup_applied:
                _, predicted = torch.max(outputs.data, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
                
            # Update progress bar
            pbar.set_postfix({
                'loss': loss.item(),
                'lr': scheduler.get_last_lr()[0]
            })
            
        # Fast validation phase - only use standard validation during training
        val_loss, val_acc, val_auc = evaluate_model(model, val_loader, criterion, use_tta=False)
        
        # Calculate training accuracy
        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / train_total if train_total > 0 else 0
        
        # Log metrics
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        # Print progress
        print(f"📊 Epoch {epoch+1}/{epochs}")
        print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
        
        # Log to CSV for tracking
        if not os.path.exists(EXPERIMENT_LOG):
            with open(EXPERIMENT_LOG, 'w') as f:
                f.write('epoch,train_loss,val_loss,train_acc,val_acc,val_auc\n')
                
        with open(EXPERIMENT_LOG, 'a') as f:
            f.write(f"{epoch+1},{train_loss:.6f},{val_loss:.6f},{train_acc:.6f},{val_acc:.6f},{val_auc:.6f}\n")
        
        # Check early stopping
        early_stopping(val_loss, val_acc, model)
        if early_stopping.early_stop:
            print(f"⚠️ Early stopping at epoch {epoch+1}")
            break
    
    print("✅ Training complete!")
    
    # Final evaluation with Test-Time Augmentation
    print("🔍 Performing final evaluation with Test-Time Augmentation...")
    model.load_state_dict(torch.load(NEW_MODEL_PATH))  # Load best model
    val_loss, val_acc, val_auc = evaluate_model(
        model, 
        val_loader, 
        criterion, 
        use_tta=True,
        save_confusion=True
    )
    print(f"📊 Final Model Results with TTA:")
    print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
    
    return history

# Visualization functions
def plot_training_history(history):
    """Plot the training and validation metrics."""
    epochs = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(16, 6))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs, history['val_loss'], label='Validation Loss', marker='s')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Accuracy', marker='o')
    plt.plot(epochs, history['val_acc'], label='Validation Accuracy', marker='s')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history14.png')
    plt.close()
    
    # Plot AUC
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['val_auc'], label='Validation AUC', marker='s', color='green')
    plt.title('Validation AUC')
    plt.xlabel('Epochs')
    plt.ylabel('AUC')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('validation_auc14.png')
    plt.close()

# Predict function with TTA
def predict_with_tta(model, image_path):
    """Make predictions with test-time augmentation."""
    model.eval()
    
    # Load and preprocess the image
    image = Image.open(image_path).convert('RGB')
    
    # Apply all TTA transforms
    tta_tensors = [transform(image).unsqueeze(0).to(DEVICE) for transform in tta_transforms]
    
    # Aggregate predictions
    probs_avg = torch.zeros((1, num_classes)).to(DEVICE)
    
    with torch.no_grad():
        for img_tensor in tta_tensors:
            with torch.cuda.amp.autocast():  # Use AMP for consistency
                outputs = model(img_tensor)
                probs = torch.softmax(outputs, dim=1)
                probs_avg += probs
    
    # Average the probabilities
    probs_avg /= len(tta_transforms)
    
    # Get predicted class and confidence
    confidence, pred_class = torch.max(probs_avg, 1)
    
    return val_dataset.classes[pred_class.item()], confidence.item()

# Initialize early stopping
early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

# Print model info
print("🔍 Model architecture:")
print(model)

print(f"🌿 Training on {len(train_dataset)} images, validating on {len(val_dataset)} images")
print(f"💻 Using device: {DEVICE}")

# Train the model
history = train_model

# Train the model
history = train_model(model, criterion, optimizer, scheduler, early_stopping, epochs=EPOCHS)

# Plot training history
plot_training_history(history)

# Load the best model for final evaluation
print(f"📥 Loading best model from {NEW_MODEL_PATH}")
model.load_state_dict(torch.load(NEW_MODEL_PATH))

# Final evaluation with Test-Time Augmentation
print("🔍 Performing final evaluation with Test-Time Augmentation...")
val_loss, val_acc, val_auc = evaluate_model(
    model, 
    val_loader, 
    criterion, 
    use_tta=True,
    save_confusion=True
)

print(f"📊 Final Model Results with TTA:")
print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
    
return history
# Example of making predictions with TTA
print("🔍 Example prediction with TTA:")
try:
    # Get a random image from validation folder
    val_dir = r"C:\Users\Asus TUF -PC\LeafSense AI training\LeafSenseProcessed\val"
    class_folders = [os.path.join(val_dir, folder) for folder in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, folder))]
    random_class = random.choice(class_folders)
    random_image = os.path.join(random_class, random.choice(os.listdir(random_class)))
    
    class_name, confidence = predict_with_tta(model, random_image)
    
    print(f"   Image: {os.path.basename(random_image)}")
    print(f"   True class: {os.path.basename(os.path.dirname(random_image))}")
    print(f"   Predicted class: {class_name} with confidence: {confidence:.4f}")
except Exception as e:
    print(f"Error making prediction: {e}")

print("✅ All done! Training and evaluation complete.")

C:\Users\Asus TUF -PC\AppData\Local\Temp\ipykernel_20632\1242488011.py:337: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(SOURCE_MODEL_PATH)

📥 Loading checkpoint from mobilenetv3_best_accuracy_improved11.pth
✅ Checkpoint loaded successfully
🔍 Model architecture:
LeafClassifier(
  (model): MobileNetV3(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)

Epoch 1/300:   0%|          | 0/509 [00:00<?, ?it/s]